# 2024–2026 Truth Social Data Cleaning for Out-of-Sample Evaluation

<p><i>Author:</i> Angelica Vanti</p>

<p><i>Project:</i> Predicting EUR/USD Movements Using Geopolitical and Macroeconomic Variables</p>

## Purpose

This notebook prepares a new sample of Donald Trump's Truth Social posts for genuine out-of-sample evaluation of the geopolitical-risk forecasting framework.

The original geopolitical indices and VAR-X models were developed using Donald Trump's first-presidency Twitter data from 2017–2021. The observations processed here occur entirely outside that estimation period and were therefore unseen during model development.

The cleaned Truth Social posts will subsequently be scored using the same LLM geopolitical-risk framework used in the original analysis. The resulting Trade Hostility, Sanctions Threat and Federal Reserve Pressure indices will then be combined with contemporaneous macroeconomic and EUR/USD observations to evaluate model performance over a later period.

This notebook therefore performs data cleaning only. No market outcomes from the out-of-sample period are used when preparing the textual observations.

In [1]:
import pandas as pd
import numpy as np
import re
import html
from pathlib import Path

## 1. Load and Inspect Raw Data

The raw Truth Social dataset is first loaded and inspected to establish its structure and temporal coverage before any cleaning is performed.

Initial checks examine the available variables, number of observations, missing values and date range. Duplicate posts and other potential data-quality issues are assessed separately before any observations are removed.

In [2]:
# Load raw Truth Social data

file_path = "../data/trump/truthsocial/trump_truths.csv"

truths = pd.read_csv(file_path)

print("Dataset shape:", truths.shape)
display(truths.head())

Dataset shape: (7310, 7)


,id,date,text,url,favorites,retweets,replies
0,113404826487996526,2024-11-01T00:18:46.050Z,Kamala has spent the final week of her failing...,https://truthsocial.com/@realDonaldTrump/11340...,13885,3748,725
1,113404838425751868,2024-11-01T00:21:48.206Z,"Just days ago, a young USMC veteran named Nich...",https://truthsocial.com/@realDonaldTrump/11340...,14641,4385,809
2,113404894982236716,2024-11-01T00:36:11.189Z,"GET OUT AND VOTE, NEVADA!!!NEVADA VOTING INFOR...",https://truthsocial.com/@realDonaldTrump/11340...,15641,3812,809
3,113405441290422074,2024-11-01T02:55:07.193Z,It was hardworking Patriots like you who built...,https://truthsocial.com/@realDonaldTrump/11340...,17652,4134,817
4,113405998200625584,2024-11-01T05:16:44.966Z,"A GREAT DAY IN NEW MEXICO, NEVADA, AND ARIZONA...",https://truthsocial.com/@realDonaldTrump/11340...,18405,4562,888


In [3]:
# Initial inspection of the raw dataset

print("Dataset information:")
truths.info()

print("\nColumns:")
print(truths.columns.tolist())

print("\nMissing values:")
print(truths.isna().sum())

print("\nExact duplicate rows:", truths.duplicated().sum())

print("Duplicate post IDs:", truths["id"].duplicated().sum())

print(
    "Empty text observations:",
    truths["text"].fillna("").str.strip().eq("").sum()
)

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 7310 entries, 0 to 7309
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         7310 non-null   int64
 1   date       7310 non-null   str  
 2   text       7310 non-null   str  
 3   url        7310 non-null   str  
 4   favorites  7310 non-null   int64
 5   retweets   7310 non-null   int64
 6   replies    7310 non-null   int64
dtypes: int64(4), str(3)
memory usage: 399.9 KB

Columns:
['id', 'date', 'text', 'url', 'favorites', 'retweets', 'replies']

Missing values:
id           0
date         0
text         0
url          0
favorites    0
retweets     0
replies      0
dtype: int64

Exact duplicate rows: 0
Duplicate post IDs: 0
Empty text observations: 0


In [4]:
# Check for repeated post text

duplicate_text = truths[
    truths.duplicated(subset=["text"], keep=False)
].sort_values(["text", "date"])

print("Number of observations with repeated text:", len(duplicate_text))
print("Number of unique texts that are repeated:",
      duplicate_text["text"].nunique())

display(
    duplicate_text[["id", "date", "text"]].head(30)
)

Number of observations with repeated text: 515
Number of unique texts that are repeated: 218


,id,date,text
6106,116202053313332968,2026-03-10T00:30:20.135Z,52% of US Likely Voters support Trump’s decisi...
6115,116207821606085249,2026-03-11T00:57:17.335Z,52% of US Likely Voters support Trump’s decisi...
294,113612461546074867,2024-12-07T16:23:05.608Z,"A GREAT HONOR, THANK YOU!"
1188,114077558611438250,2025-02-27T19:43:23.617Z,"A GREAT HONOR, THANK YOU!"
5180,115850410088588555,2026-01-06T22:02:50.342Z,"A Great and Highly Respected Hispanic Judge, v..."
5798,116083520444818787,2026-02-17T02:05:52.102Z,"A Great and Highly Respected Hispanic Judge, v..."
5953,116143518815107485,2026-02-27T16:24:14.611Z,"A Great and Highly Respected Hispanic Judge, v..."
3909,115253901757845391,2025-09-23T13:42:55.552Z,After reviewing the details of the unserious a...
3916,115260622454907511,2025-09-24T18:12:05.289Z,After reviewing the details of the unserious a...
6495,116400914571664797,2026-04-14T03:23:22.168Z,Alan Dershowitz: Trump could move to expunge 2...


In [5]:
# Examine how close together repeated posts occur

duplicate_text = truths[
    truths.duplicated(subset=["text"], keep=False)
].copy()

# Force date conversion here
duplicate_text["date"] = pd.to_datetime(
    duplicate_text["date"],
    errors="coerce",
    utc=True
)

# Sort repeated posts by text and date
duplicate_text = duplicate_text.sort_values(["text", "date"])

# Find previous occurrence of the same text
duplicate_text["previous_date"] = (
    duplicate_text.groupby("text")["date"].shift(1)
)

# Calculate gap between repeated posts
duplicate_text["gap_hours"] = (
    duplicate_text["date"] - duplicate_text["previous_date"]
).dt.total_seconds() / 3600

print(
    "Repeated posts within 24 hours:",
    duplicate_text["gap_hours"].le(24).sum()
)

print(
    "Repeated posts within 7 days:",
    duplicate_text["gap_hours"].le(24 * 7).sum()
)

display(
    duplicate_text[
        duplicate_text["gap_hours"].notna()
    ][
        ["date", "previous_date", "gap_hours", "text"]
    ]
    .sort_values("gap_hours")
    .head(30)
)

Repeated posts within 24 hours: 35
Repeated posts within 7 days: 98


,date,previous_date,gap_hours,text
6802,2026-05-06 12:29:21.011000+00:00,2026-05-06 12:29:20.869000+00:00,0.000039,Top DOJ official predicts Supreme Court will d...
2019,2025-04-28 21:16:17.903000+00:00,2025-04-28 21:16:10.424000+00:00,0.002077,"As we reach our Historic First 100 Days, I am ..."
7050,2026-05-24 12:12:27.661000+00:00,2026-05-24 12:12:16.821000+00:00,0.003011,“President Xi and President Trump are AMAZING!”
989,2025-02-15 17:48:22.686000+00:00,2025-02-15 17:48:07.397000+00:00,0.004247,https://www.washingtonexaminer.com/news/world/...
4680,2025-11-26 00:35:22.651000+00:00,2025-11-26 00:34:37.268000+00:00,0.012606,https://truthsocial.com/@IStandWithTrump47/115...
6485,2026-04-13 06:37:44.521000+00:00,2026-04-13 06:35:51.423000+00:00,0.031416,Impeachment Bombshell: Secret memos expose Ukr...
3628,2025-08-25 16:46:43.527000+00:00,2025-08-25 16:44:38.837000+00:00,0.034636,https://www.whitehouse.gov/presidential-action...
2126,2025-05-06 00:46:29.061000+00:00,2025-05-06 00:43:34.225000+00:00,0.048566,RT: https://truthsocial.com/users/realDonaldTr...
3217,2025-07-30 11:35:12.124000+00:00,2025-07-30 11:31:47.845000+00:00,0.056744,https://truthsocial.com/users/SpiritualStreetf...
1189,2025-02-27 20:10:51.497000+00:00,2025-02-27 19:41:00.103000+00:00,0.497609,RT: https://truthsocial.com/users/realDonaldTr...


### Repeated Text Diagnostic

The dataset contains repeated post text, including a small number of identical posts published within short time intervals.

Because repeated wording may represent either deliberate republication or duplicated content and the available dataset does not provide a reliable structural indicator distinguishing the two, repeated posts are not removed solely on the basis of identical text or temporal proximity.

Only exact duplicate records or duplicate post identifiers are treated as duplicates. This avoids introducing an arbitrary time-based removal rule into the out-of-sample preprocessing procedure.

In [6]:
# Identify link-only posts 

url_pattern = r'https?://\S+|www\.\S+'

# Remove URLs temporarily to test whether substantive text remains
text_without_urls = (
    truths["text"]
    .str.replace(url_pattern, "", regex=True)
    .str.strip()
)

link_only = truths[text_without_urls.eq("")]

print("Link-only posts:", len(link_only))

display(
    link_only[["id", "date", "text"]].head(20)
)


Link-only posts: 1400


,id,date,text
7,113408744364085184,2024-11-01T16:55:08.094Z,https://nypost.com/2024/11/01/us-news/election...
49,113417310183513805,2024-11-03T05:13:32.126Z,https://www.breitbart.com/clips/2009/10/05/pis...
50,113417335829985714,2024-11-03T05:20:03.463Z,https://www.breitbart.com/the-media/2024/11/02...
53,113417360552033273,2024-11-03T05:26:20.688Z,https://www.breitbart.com/clips/2024/11/01/pel...
56,113419592954504417,2024-11-03T14:54:04.447Z,https://www.rsbnetwork.com/news/final-trump-ra...
68,113421683921327580,2024-11-03T23:45:50.073Z,https://www.DonaldJTrump.com
76,113425266314029290,2024-11-04T14:56:53.047Z,https://swampthevoteusa.com/
88,113427332561913372,2024-11-04T23:42:21.487Z,https://justthenews.com/politics-policy/electi...
99,113431692663654098,2024-11-05T18:11:11.361Z,https://protectthevote.com/
102,113432003355510692,2024-11-05T19:30:12.141Z,https://justthenews.com/politics-policy/friele...


### Removal of Link-Only Posts

Posts containing only a URL and no substantive textual content are removed because the geopolitical-risk annotation procedure operates on the written content of each post. Posts containing both substantive text and a URL are retained; only observations for which removing the URL leaves no textual content are excluded.

This applies the same text-based filtering principle used for the original Twitter sample.

In [7]:
# Remove link-only posts

n_before = len(truths)

url_pattern = r'https?://\S+|www\.\S+'

text_without_urls = (
    truths["text"]
    .str.replace(url_pattern, "", regex=True)
    .str.strip()
)

truths = truths.loc[~text_without_urls.eq("")].copy()
truths = truths.reset_index(drop=True)

n_removed = n_before - len(truths)

print("Observations before:", n_before)
print("Link-only posts removed:", n_removed)
print("Observations remaining:", len(truths))

Observations before: 7310
Link-only posts removed: 1400
Observations remaining: 5910


In [8]:
# Identify Truth Social reposts beginning with "RT:"

repost_mask = truths["text"].str.match(
    r"^\s*RT:\s*https?://",
    case=False,
    na=False
)

print("Truth Social reposts identified:", repost_mask.sum())

display(
    truths.loc[
        repost_mask,
        ["id", "date", "text"]
    ].head(30)
)

Truth Social reposts identified: 670


,id,date,text
47,113416835682642733,2024-11-03T03:12:51.819Z,RT: https://truthsocial.com/users/realDonaldTr...
66,113423151971796161,2024-11-04T05:59:10.745Z,RT: https://truthsocial.com/users/realDonaldTr...
87,113428485840736779,2024-11-05T04:35:39.126Z,RT: https://truthsocial.com/users/TeamTrump/st...
93,113432133120615756,2024-11-05T20:03:12.198Z,RT: https://truthsocial.com/users/realDonaldTr...
120,113464580711973698,2024-11-11T13:35:03.152Z,RT: https://truthsocial.com/users/AngelaQPatri...
124,113466181286518065,2024-11-11T20:22:05.983Z,RT: https://truthsocial.com/users/alx/statuses...
147,113503150672865350,2024-11-18T09:03:54.047Z,RT: https://truthsocial.com/users/TomFitton/st...
245,113609405236508791,2024-12-07T03:25:50.023Z,RT: https://truthsocial.com/users/realDonaldTr...
249,113612489566973987,2024-12-07T16:30:13.169Z,RT: https://truthsocial.com/users/realDonaldTr...
251,113613535242740192,2024-12-07T20:56:08.915Z,RT: https://truthsocial.com/users/realDonaldTr...


In [9]:
# Remove Truth Social reposts

n_before = len(truths)

truths = truths.loc[~repost_mask].copy()
truths = truths.reset_index(drop=True)

n_removed = n_before - len(truths)

print("Observations before:", n_before)
print("Reposts removed:", n_removed)
print("Observations remaining:", len(truths))

Observations before: 5910
Reposts removed: 670
Observations remaining: 5240


## Text Cleaning

The remaining posts are cleaned conservatively before annotation. The objective is to remove formatting artefacts while preserving the substantive wording and meaning of each post.

URLs embedded within otherwise substantive posts are removed, since the annotation model evaluates the textual content and does not access external webpages. Excess whitespace is also normalised. No stemming, stop-word removal, lowercasing, or other aggressive text preprocessing is applied, as these transformations could remove contextual information relevant to geopolitical-risk classification.

In [10]:
# Clean textual content while preserving substantive wording

truths["text_clean"] = (
    truths["text"]
    # Remove URLs from posts that otherwise contain text
    .str.replace(r"https?://\S+|www\.\S+", "", regex=True)
    # Replace line breaks/tabs with spaces
    .str.replace(r"[\r\n\t]+", " ", regex=True)
    # Collapse repeated whitespace
    .str.replace(r"\s+", " ", regex=True)
    # Remove leading/trailing whitespace
    .str.strip()
)

print(
    "Empty observations after text cleaning:",
    truths["text_clean"].eq("").sum()
)

display(
    truths[["id", "date", "text", "text_clean"]].head(20)
)

Empty observations after text cleaning: 0


,id,date,text,text_clean
0,113404826487996526,2024-11-01T00:18:46.050Z,Kamala has spent the final week of her failing...,Kamala has spent the final week of her failing...
1,113404838425751868,2024-11-01T00:21:48.206Z,"Just days ago, a young USMC veteran named Nich...","Just days ago, a young USMC veteran named Nich..."
2,113404894982236716,2024-11-01T00:36:11.189Z,"GET OUT AND VOTE, NEVADA!!!NEVADA VOTING INFOR...","GET OUT AND VOTE, NEVADA!!!NEVADA VOTING INFOR..."
3,113405441290422074,2024-11-01T02:55:07.193Z,It was hardworking Patriots like you who built...,It was hardworking Patriots like you who built...
4,113405998200625584,2024-11-01T05:16:44.966Z,"A GREAT DAY IN NEW MEXICO, NEVADA, AND ARIZONA...","A GREAT DAY IN NEW MEXICO, NEVADA, AND ARIZONA..."
5,113406381620790894,2024-11-01T06:54:15.500Z,"“The Trump Economic Miracle,” a new book by th...","“The Trump Economic Miracle,” a new book by th..."
6,113406435291253643,2024-11-01T07:07:54.442Z,"“MELANIA,” the new book just out by our great ...","“MELANIA,” the new book just out by our great ..."
7,113408902359347846,2024-11-01T17:35:18.911Z,Wishing everyone a Blessed and Happy All Saint...,Wishing everyone a Blessed and Happy All Saint...
8,113409094351404093,2024-11-01T18:24:08.477Z,"Mark Cuban, a total loser, is being decimated ...","Mark Cuban, a total loser, is being decimated ..."
9,113409095736564025,2024-11-01T18:24:29.613Z,All I’m saying about Liz Cheney is that she is...,All I’m saying about Liz Cheney is that she is...


In [11]:
# Final sanity checks

print("Final number of observations:", len(truths))

print("\nDate range:")
print("Start:", truths["date"].min())
print("End:  ", truths["date"].max())

print("\nMissing cleaned text:", truths["text_clean"].isna().sum())
print("Empty cleaned text:", truths["text_clean"].eq("").sum())

print("\nDuplicate post IDs:", truths["id"].duplicated().sum())

print("\nCleaned text length:")
print(truths["text_clean"].str.len().describe())

# Inspect the shortest remaining posts
display(
    truths[
        ["id", "date", "text_clean"]
    ]
    .assign(text_length=truths["text_clean"].str.len())
    .sort_values("text_length")
    .head(30)
)

Final number of observations: 5240

Date range:
Start: 2024-11-01T00:18:46.050Z
End:   2026-06-11T01:21:45.697Z

Missing cleaned text: 0
Empty cleaned text: 0

Duplicate post IDs: 0

Cleaned text length:
count    5240.000000
mean      432.035496
std       439.680516
min         2.000000
25%        98.000000
50%       282.000000
75%       656.000000
max      4183.000000
Name: text_clean, dtype: float64


,id,date,text_clean,text_length
938,114186080634823311,2025-03-18T23:41:58.285Z,🇺🇸,2
2351,115044351128754901,2025-08-17T13:31:26.748Z,Bela,4
756,114061717963482741,2025-02-25T00:34:54.512Z,🇺🇸🇫🇷,4
752,114060894274400497,2025-02-24T21:05:26.014Z,🇺🇸🇫🇷,4
1507,114524964544595396,2025-05-17T20:04:36.456Z,🇸🇦🇺🇸,4
1508,114524968005625641,2025-05-17T20:05:29.270Z,🇶🇦🇺🇸,4
1509,114524979790255530,2025-05-17T20:08:29.007Z,🇦🇪🇺🇸,4
881,114151032918411177,2025-03-12T19:08:52.642Z,🇺🇸🇮🇪,4
1433,114475204336988426,2025-05-09T01:09:55.944Z,🇺🇸🇬🇧,4
3476,115822915162697186,2026-01-02T01:30:31.145Z,WOW!,4


In [12]:
# Inspect very short remaining posts

short_posts = truths[
    truths["text_clean"].str.len() <= 20
][["id", "date", "text", "text_clean"]]

print("Posts with 20 characters or fewer:", len(short_posts))

pd.set_option("display.max_colwidth", None)
display(short_posts)

Posts with 20 characters or fewer: 106


,id,date,text,text_clean
54,113420527789498274,2024-11-03T18:51:48.897Z,I LOVE TRUTH SOCIAL!,I LOVE TRUTH SOCIAL!
62,113421835789140768,2024-11-04T00:24:27.393Z,#Unity2024 #MAGA https://swampthevoteusa.com/,#Unity2024 #MAGA
82,113428019736332599,2024-11-05T02:37:06.934Z,"THANK YOU, JOE!","THANK YOU, JOE!"
144,113509313119316302,2024-11-19T11:11:05.520Z,“CHANGE IS COMING”,“CHANGE IS COMING”
196,113591029172090604,2024-12-03T21:32:33.530Z,Oh Canada!,Oh Canada!
...,...,...,...,...
4895,116564114846189487,2026-05-12T23:07:20.696Z,I WANT YOU!,I WANT YOU!
4906,116580615618774330,2026-05-15T21:03:42.540Z,FREE TINA!,FREE TINA!
5010,116624933679947822,2026-05-23T16:54:22.450Z,Won Big! https://www.washingtonexaminer.com/in_focus/4571185/thomas-massie-useless-tenure-congress-coming-to-an-end/,Won Big!
5107,116677910570104917,2026-06-02T01:27:05.640Z,51st State! https://www.bloomberg.com/news/articles/2026-05-29/canada-dips-into-technical-recession-for-first-time-since-2020,51st State!


## Final Dataset

Following preprocessing, the dataset contains 5,240 original textual Truth Social posts. Link-only observations and posts explicitly identified as reposts were removed, while repeated posts with distinct identifiers and timestamps were retained.

Text cleaning was deliberately conservative to preserve information relevant to geopolitical-risk annotation. URLs and formatting artefacts were removed, while the substantive wording, punctuation and capitalisation of each post were retained.

The final sample spans 1 November 2024 to 11 June 2026 and contains no missing or empty cleaned-text observations and no duplicate post identifiers.

In [13]:
# Prepare final cleaned dataset in the same format as cleaned_tweets.csv

truths_final = truths[
    ["id", "date", "text_clean"]
].copy()

# Convert date to the same format as the original Twitter dataset
truths_final["Date"] = (
    pd.to_datetime(truths_final["date"], utc=True)
    .dt.tz_localize(None)
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)

# Match the original Twitter column names
truths_final["Tweet ID"] = truths_final["id"].astype(str)
truths_final["Tweet"] = truths_final["text_clean"]

# Each Truth is treated as a standalone post
truths_final["Tweet IDs"] = truths_final["Tweet ID"].apply(
    lambda x: str([x])
)
truths_final["Thread_end"] = truths_final["Date"]
truths_final["Thread_length"] = 1

# Match the original cleaned_tweets.csv column order
truths_final = truths_final[
    [
        "Tweet ID",
        "Tweet IDs",
        "Tweet",
        "Date",
        "Thread_end",
        "Thread_length"
    ]
]

print("Final dataset shape:", truths_final.shape)
print("Columns:", truths_final.columns.tolist())
display(truths_final.head())

Final dataset shape: (5240, 6)
Columns: ['Tweet ID', 'Tweet IDs', 'Tweet', 'Date', 'Thread_end', 'Thread_length']


,Tweet ID,Tweet IDs,Tweet,Date,Thread_end,Thread_length
0,113404826487996526,['113404826487996526'],"Kamala has spent the final week of her failing campaign comparing her political opponents to the most evil mass murderers in history. Two days ago, Joe Biden called our supporters garbage…You can’t LEAD America if you don’t LOVE Americans!",2024-11-01 00:18:46,2024-11-01 00:18:46,1
1,113404838425751868,['113404838425751868'],"Just days ago, a young USMC veteran named Nicholas Quets from Arizona was driving through Mexico for a beach weekend with his friends when he was viciously gunned down on the highway and murdered by members of a Mexican Cartel. His family joined me this evening in Henderson, Nevada…",2024-11-01 00:21:48,2024-11-01 00:21:48,1
2,113404894982236716,['113404894982236716'],"GET OUT AND VOTE, NEVADA!!!NEVADA VOTING INFORMATION HERE:",2024-11-01 00:36:11,2024-11-01 00:36:11,1
3,113405441290422074,['113405441290422074'],"It was hardworking Patriots like you who built this Country, and 5 days from now, it is hardworking Patriots like you who are going to SAVE our Country! THANK YOU, NEVADA!",2024-11-01 02:55:07,2024-11-01 02:55:07,1
4,113405998200625584,['113405998200625584'],"A GREAT DAY IN NEW MEXICO, NEVADA, AND ARIZONA—THANK YOU! With your VOTE this November, we are going to Fire Kamala, and we are going to SAVE AMERICA!",2024-11-01 05:16:44,2024-11-01 05:16:44,1


In [14]:
# Export final cleaned Truth Social dataset
# in the same structure as the original Twitter LLM input

output_path = "../data/processed/truths_cleaned.csv"

truths_final.to_csv(output_path, index=False)

print(f"Saved cleaned Truth Social dataset to: {output_path}")
print("Exported observations:", len(truths_final))
print("Exported columns:", truths_final.columns.tolist())

Saved cleaned Truth Social dataset to: ../data/processed/truths_cleaned.csv
Exported observations: 5240
Exported columns: ['Tweet ID', 'Tweet IDs', 'Tweet', 'Date', 'Thread_end', 'Thread_length']


In [15]:
# Final compatibility check against original Twitter LLM input

original_tweets = pd.read_csv("../data/processed/cleaned_tweets.csv")

print("Original Twitter columns:")
print(original_tweets.columns.tolist())

print("\nTruth Social columns:")
print(truths_final.columns.tolist())

print(
    "\nColumns match exactly:",
    original_tweets.columns.tolist() == truths_final.columns.tolist()
)

print("\n--- Dataset sizes ---")
print("Original Twitter:", original_tweets.shape)
print("Truth Social:", truths_final.shape)

print("\n--- Missing values in Truth Social ---")
print(truths_final.isna().sum())

print("\n--- First two Truth Social observations ---")
display(truths_final.head(2))

Original Twitter columns:
['Tweet ID', 'Tweet IDs', 'Tweet', 'Date', 'Thread_end', 'Thread_length']

Truth Social columns:
['Tweet ID', 'Tweet IDs', 'Tweet', 'Date', 'Thread_end', 'Thread_length']

Columns match exactly: True

--- Dataset sizes ---
Original Twitter: (15284, 6)
Truth Social: (5240, 6)

--- Missing values in Truth Social ---
Tweet ID         0
Tweet IDs        0
Tweet            0
Date             0
Thread_end       0
Thread_length    0
dtype: int64

--- First two Truth Social observations ---


,Tweet ID,Tweet IDs,Tweet,Date,Thread_end,Thread_length
0,113404826487996526,['113404826487996526'],"Kamala has spent the final week of her failing campaign comparing her political opponents to the most evil mass murderers in history. Two days ago, Joe Biden called our supporters garbage…You can’t LEAD America if you don’t LOVE Americans!",2024-11-01 00:18:46,2024-11-01 00:18:46,1
1,113404838425751868,['113404838425751868'],"Just days ago, a young USMC veteran named Nicholas Quets from Arizona was driving through Mexico for a beach weekend with his friends when he was viciously gunned down on the highway and murdered by members of a Mexican Cartel. His family joined me this evening in Henderson, Nevada…",2024-11-01 00:21:48,2024-11-01 00:21:48,1
